# QFT repeated secret

**Run All** to see whether Bob alarms before two Eves recover the same four-bit secret. The first figure compares the two detectors. The second shows how confidence in each secret bit changes with repeated transmissions.

## Configuration

In [ ]:
ATTEMPTS = 100  # Cap on transmissions of the four-bit secret.
TRIALS = 256  # Independent trials each for fit, held-out noise, and attack.
BLOCK_QUBITS = 2  # Choose 2 or 4. Half of each block is fresh hidden checks.
INTERCEPT_PROBABILITY = 0.025  # Per Eve, per qubit, on each leg.
EVE_Z_PROBABILITIES = (0.1, 0.9)  # The two Eves favor different bases.
Q_Z, Q_X = 0.002, 0.038  # Per-leg Pauli X and Z probabilities.
SEED = 20261122

## 1. Does Bob alarm before recovery?

Both detectors see the same transmissions. The left panel checks their false alarms on independent noise-only trials. The right panel shows which event comes first during an attack. Recovery requires the correct four-bit secret to reach 99% posterior confidence. Bob's alarm wins a tie.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

from quantum_study.qft_study import run_qft_study

summary, curves = run_qft_study(
    attempts=ATTEMPTS, trials=TRIALS, block_qubits=BLOCK_QUBITS,
    intercept_probability=INTERCEPT_PROBABILITY,
    eve_z_probabilities=EVE_Z_PROBABILITIES, q_z=Q_Z, q_x=Q_X, seed=SEED,
)
summary = summary.set_index("detector")
outcomes = summary[[
    "alarm_rate", "correct_recovery_rate", "incorrect_recovery_rate", "inconclusive_rate",
]].copy()
outcomes.columns = ["Bob alarm", "Correct recovery", "Incorrect declaration", "Reached cap"]

fig, (null_ax, attack_ax) = plt.subplots(1, 2, sharey=True, figsize=(10, 3.5), layout="constrained")
false_alarms = summary["false_alarm_rate"]
bars = null_ax.barh(summary.index, false_alarms, height=0.5, color="#4C78A8")
null_ax.bar_label(bars, labels=[f"{rate:.1%}" for rate in false_alarms], padding=4,
                  bbox={"facecolor": "white", "edgecolor": "none", "pad": 0.2})
null_ax.set_xlim(0, max(0.08, false_alarms.max() * 1.25))
null_ax.axvline(0.05, color="0.3", linestyle="--")
null_ax.text(0.05, 0.98, "5% target", transform=null_ax.get_xaxis_transform(),
             ha="center", va="top", fontsize=9, backgroundcolor="white")
null_ax.set(title="Noise only · false alarms", xlabel="Fraction of trials", ylabel="")
outcomes.plot.barh(ax=attack_ax, stacked=True, color=["#4C78A8", "#14866D", "#D56C36", "#C9D4DB"])
attack_ax.set(title="Interception · first outcome", xlabel="Fraction of trials", ylabel="", xlim=(0, 1))
attack_ax.tick_params(labelleft=False)
alarm_labels = [f"{rate:.1%}" if rate >= 0.08 else "" for rate in outcomes["Bob alarm"]]
attack_ax.bar_label(attack_ax.containers[0], labels=alarm_labels, label_type="center", color="white")
attack_ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.23), ncol=2)
for axis in (null_ax, attack_ax):
    axis.xaxis.set_major_formatter(PercentFormatter(1))
plt.show()

for detector, row in summary.iterrows():
    recovered = round(row["correct_recovery_rate"] * TRIALS)
    print(f"{detector}: {recovered}/{TRIALS} correct declarations before Bob's alarm.")

## 2. What do the Eves learn about each bit?

The curves show the average probability the coalition assigns to each **true** bit value. The axis zooms in on deviations from the 50% uninformed guess. A trial's confidence stays at its last value once Bob alarms or Eve declares, so stopped trials still count in the average.

In [ ]:
bits = curves[[f"bit_{bit}" for bit in range(1, 5)]]
lower = max(0, min(0.48, bits.min().min() - 0.02))
upper = min(1, max(0.55, bits.max().max() + 0.02))
fig, axes = plt.subplots(1, 2, sharey=True, figsize=(10, 3.4), layout="constrained")
for axis, (detector, group) in zip(axes, curves.groupby("detector", sort=False)):
    for bit in range(1, 5):
        axis.plot(group["attempt"], group[f"bit_{bit}"], label=f"Bit {bit}")
    axis.axhline(0.5, color="0.5", linestyle="--", linewidth=1)
    axis.set(title=detector, xlabel="Attempt", ylim=(lower, upper))
    axis.yaxis.set_major_formatter(PercentFormatter(1))
axes[0].set_ylabel("Mean probability of true bit")
axes[-1].legend(loc="upper left", ncol=2)
plt.show()

## Method and sources

- **Circuit.** Bob scrambles private phases, Alice encodes the secret with fresh hidden checks, and Bob reverses the phases and applies the inverse QFT. [Papadopoulos and Linvill, §4.1 Eqs. (1)–(3) and §6.1](https://arxiv.org/abs/2404.12507v2)
- **Experiment.** The proposal's §2 supplies the repeated four-bit key, per-leg Pauli model, and recovery threshold. This focused run compares pooled check errors and block check-pattern surprisal at 25%, 50%, and 100% of the attempt cap. Each cutoff targets 5% for any alarm across those looks. The reference rates come from [Erven et al., §4](https://arxiv.org/abs/0901.0960v2).
- **Inference.** Averaging Bob's uniform private phases gives an exact mixed starting state for Eve's record likelihood. Bob's calculation still uses his actual phases. [Exchange code](quantum_study/qft_exchange.py) · [Study code](quantum_study/qft_study.py)

For the separate 500-attempt recalibration chart, open the [QFT pilot notebook](archive/qft_pilot/QFT_Factorial_Pilot.ipynb) and set `RUN_RECALIBRATION_DEMO = True`.